# Train "my-ai-coder" — Qwen3.5 **9B** on Kaggle's free 2× T4 GPUs

**One-time setup (~5 minutes):**
1. kaggle.com → sign in → **Settings → Phone verification** (required for free GPU + internet)
2. **Create → New Notebook → File → Import Notebook** → upload this `.ipynb` file
3. Right panel **Session options**: **Accelerator = GPU T4 x2**, **Internet = On**
4. Hugging Face: huggingface.co → Settings → Access Tokens → **Create new token**, type **Write**, copy it
5. Kaggle notebook menu **Add-ons → Secrets → Add Secret**: Label `HF_TOKEN`, Value = your token, then tick the box to attach it to this notebook

**Run it:** top right **Save Version → "Save & Run All (Commit)" → Save**.
This runs in the background — **you can close the tab and even turn off your PC.** Watch progress under your notebook's **Versions/Logs**. Expect roughly 4–8 hours (12-hour hard limit). Uses ~that many hours of your free 30 h/week.

> Why Kaggle: training the 9B needs ~22 GB of GPU memory. Kaggle's free tier gives two 16 GB GPUs (32 GB); Colab's free tier gives one 15 GB GPU.

## 1. Settings

In [ ]:
HF_USERNAME   = "Hudson2011"
LORA_REPO     = "my-ai-coder-9B-lora"   # small safety backup of the training (uploaded first)
GGUF_REPO     = "my-ai-coder-9B-GGUF"   # the file Ollama runs
BASE_MODEL    = "unsloth/Qwen3.5-9B"
NUM_EXAMPLES  = 3000     # coding examples
MAX_STEPS     = 250      # training steps; lower this if a run gets close to 12 h
MAX_SEQ_LEN   = 3072     # longest example kept (tokens)

## 2. Install Unsloth

In [ ]:
%%capture
!pip install --upgrade unsloth unsloth_zoo

In [ ]:
import torch, os
n = torch.cuda.device_count()
assert n >= 2, f"Found {n} GPU(s). Set Accelerator = GPU T4 x2 in Session options."
for i in range(n):
    print(f"GPU {i}:", torch.cuda.get_device_name(i), round(torch.cuda.get_device_properties(i).total_memory/1e9,1), "GB")
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")   # fails if the secret isn't attached (setup step 5)
print("HF token found.")

## 3. Load the 9B model, split across both GPUs

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit   = False,       # Unsloth advises against 4-bit training for Qwen3.5
    load_in_16bit  = True,
    device_map     = "balanced",  # spreads the model over both T4s
)
text_tok = getattr(tokenizer, "tokenizer", tokenizer)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16, lora_alpha = 16, lora_dropout = 0, bias = "none",
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    max_seq_length = MAX_SEQ_LEN,
)
for i in range(n): print(f"GPU {i} memory used: {torch.cuda.memory_allocated(i)/1e9:.1f} GB")

## 4. Test BEFORE training

In [ ]:
TEST_PROMPTS = [
    "Write a Python function that returns the longest palindromic substring of a string. Explain the time complexity.",
    "In JavaScript, why does `[1,2,10].sort()` give [1,10,2]? Show the fix.",
    "What causes inflation, in 3 sentences?",   # general knowledge: should NOT get worse
]
def ask(prompt, max_new_tokens=700):
    FastLanguageModel.for_inference(model)
    ids = text_tok.apply_chat_template([{"role":"user","content":prompt}], add_generation_prompt=True, return_tensors="pt").to("cuda:0")
    out = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, temperature=0.6, top_p=0.95, top_k=20)
    FastLanguageModel.for_training(model)
    return text_tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

for p in TEST_PROMPTS: print("Q:", p, "\n--- BEFORE ---\n", ask(p)[-1500:], "\n" + "="*80)

## 5. Coding training data (streamed)

In [ ]:
from datasets import load_dataset, Dataset

stream = load_dataset("nvidia/OpenCodeReasoning", "split_0", split="split_0", streaming=True)
stream = stream.shuffle(seed=3407, buffer_size=10_000)
rows = []
for ex in stream:
    msgs = [{"role":"user","content":ex["input"]}, {"role":"assistant","content":ex["output"]}]  # keeps <think> reasoning
    text = text_tok.apply_chat_template(msgs, tokenize=False)
    if len(text_tok(text).input_ids) <= MAX_SEQ_LEN:
        rows.append({"text": text})
    if len(rows) >= NUM_EXAMPLES: break
dataset = Dataset.from_list(rows)
print(len(dataset), "examples. Sample:\n", dataset[0]["text"][:600])

## 6. Train (the long part)

In [ ]:
from trl import SFTTrainer, SFTConfig
import time; t0 = time.time()
trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LEN,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = MAX_STEPS,
        learning_rate = 1e-4,           # gentle, limits forgetting
        logging_steps = 5,
        optim = "adamw_8bit",
        lr_scheduler_type = "cosine",
        fp16 = True, bf16 = False,      # T4s don't support bf16
        output_dir = "/kaggle/working/outputs",
        save_strategy = "no",
        seed = 3407,
        report_to = "none",
        dataset_num_proc = 1,
    ),
)
print(trainer.train())
print(f"Training took {(time.time()-t0)/3600:.1f} hours")

## 7. Back up the training to Hugging Face right away (small, ~100 MB)

In [ ]:
model.push_to_hub(f"{HF_USERNAME}/{LORA_REPO}", token=HF_TOKEN, private=True)
tokenizer.push_to_hub(f"{HF_USERNAME}/{LORA_REPO}", token=HF_TOKEN, private=True)
print("Backup saved: https://huggingface.co/" + f"{HF_USERNAME}/{LORA_REPO}")

## 8. Test AFTER training — compare with section 4

In [ ]:
for p in TEST_PROMPTS: print("Q:", p, "\n--- AFTER ---\n", ask(p)[-1500:], "\n" + "="*80)

## 9. Build the Ollama file (GGUF) and upload it (30–60 min)

In [ ]:
os.makedirs("/tmp/export", exist_ok=True); os.chdir("/tmp/export")   # big temp files go here, not the 20 GB output folder
model.push_to_hub_gguf(f"{HF_USERNAME}/{GGUF_REPO}", tokenizer, quantization_method="q4_k_m", token=HF_TOKEN)
print("Done: https://huggingface.co/" + f"{HF_USERNAME}/{GGUF_REPO}")

## 10. On your PC

In the `local-ai` folder, run:
```
ollama create my-ai-coder -f Modelfile.trained
ollama run my-ai-coder
```
`Modelfile.trained` pulls the GGUF from `hf.co/Hudson2011/my-ai-coder-9B-GGUF`. If the upload went into a private repo, make it public on Hugging Face first (repo → Settings → Visibility), or Ollama can't download it.